# Chicago TNP: Chronos-2 vs LightGBM Forecasting Benchmark  
# Chicago TNP：Chronos-2 与 LightGBM 客流预测基准

This notebook compares last-week demand, three Chronos-2 configurations, and two LightGBM configurations on the same hourly H3 forecast origins. Chronos runs in an isolated Python 3.11 process so model runtime failures do not stop the Jupyter kernel, and completed origins are checkpointed for reproducible resume.  
本 Notebook 在相同的小时级 H3 预测起点上，对比上周同期基线、三种 Chronos-2 配置和两种 LightGBM 配置。Chronos 在隔离的 Python 3.11 进程中运行，模型运行故障不会终止 Jupyter kernel；已完成的预测起点会保存检查点，支持可复现续跑。

## Experiment design / 实验设计

- Top 10 H3 cells are selected from 2022-2023; 2024 is the test period. / Top 10 H3 单元根据 2022-2023 选择，2024 为测试期。
- Each origin uses 28 days of history to forecast 24 hours. / 每个预测起点使用过去 28 天预测未来 24 小时。
- Chronos and LightGBM use identical origins, targets, and horizon alignment: horizon 1 is the forecast origin and horizon 24 is origin + 23 hours. / Chronos 与 LightGBM 使用完全相同的起点、目标和跨度定义：horizon 1 为预测起点，horizon 24 为起点后 23 小时。
- Set `CHICAGO_TNP_PROJECT_DIR` and the `MATRIXONE_*` environment variables before execution. / 运行前配置项目目录与 MatrixOne 环境变量。


## 0. Install dependencies / 安装依赖

Use a Python 3.11 kernel for this notebook. Chronos and LightGBM are installed in the same environment, while Chronos inference runs in a separate process.  
本 Notebook 使用 Python 3.11 kernel。Chronos 与 LightGBM 安装在同一环境中，但 Chronos 推理在独立进程内运行。


In [ ]:
# Cell 0 - Install dependencies in Python 3.11 / 在 Python 3.11 中安装依赖
%pip install -q pandas numpy matplotlib scikit-learn pymysql holidays requests certifi pyarrow joblib lightgbm "chronos-forecasting>=2.1,<3" torch


In [ ]:
# Cell 1 - Verify Python 3.11 and import packages / 验证 Python 3.11 并导入包
# 主进程不导入 torch 或 chronos

import sys
import os
import json
import time
import gc
import warnings
import subprocess
from pathlib import Path
from datetime import date, timedelta
import calendar

if sys.version_info[:2] != (3, 11):
    raise RuntimeError(
        "This notebook must use a Python 3.11 kernel. "
        f"Current Python: {sys.version}"
    )

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pymysql
import holidays
import requests
import certifi
import joblib

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    precision_score,
    recall_score,
    f1_score,
    average_precision_score,
)

print("Python executable:", sys.executable)
print("Python version:", sys.version)
print("Main process PID:", os.getpid())
print("PyTorch is intentionally not imported in the Jupyter process.")


In [ ]:
import os
from getpass import getpass
# Cell 2 - Settings and MatrixOne connection / 参数与 MatrixOne 连接

# Portable project configuration / 可移植项目配置
PROJECT_DIR = Path(
    os.getenv("CHICAGO_TNP_PROJECT_DIR", str(Path.cwd()))
).expanduser().resolve()

OUTPUT_DIR = PROJECT_DIR / "notebook_outputs_chronos2_isolated_vs_lightgbm"
CACHE_DIR = OUTPUT_DIR / "cache"
WORKER_DIR = OUTPUT_DIR / "chronos_worker"
MODEL_DIR = OUTPUT_DIR / "models"

for folder in [OUTPUT_DIR, CACHE_DIR, WORKER_DIR, MODEL_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

MO_HOST = os.getenv("MATRIXONE_HOST", "127.0.0.1")
MO_PORT = int(os.getenv("MATRIXONE_PORT", "6001"))
MO_USER = os.getenv("MATRIXONE_USER", "root")
MO_PASSWORD = os.getenv("MATRIXONE_PASSWORD") or getpass("MatrixOne password / MatrixOne 密码: ")
MO_DB = os.getenv("MATRIXONE_DATABASE", "chicago_tnp")
ANALYSIS_TABLE = "unified_trips_h3_res9"

DATA_START = pd.Timestamp("2022-01-01 00:00:00")
DATA_END = pd.Timestamp("2024-12-31 23:00:00")
TRAIN_END = pd.Timestamp("2024-01-01 00:00:00")

TOP_N_CELLS = 10
CONTEXT_LENGTH = 24 * 28
PREDICTION_LENGTH = 24
BACKTEST_STRIDE_HOURS = 73
ML_TRAIN_ORIGIN_STRIDE_HOURS = 6

CHRONOS_MODEL_ID = "autogluon/chronos-2-small"
CHRONOS_DEVICE = "cpu"
CHRONOS_THREADS = 1

HIGH_Z_THRESHOLD = 3.0
MIN_PEAK_COUNT = 100
MIN_EXPECTED_FOR_PEAK = 30

BOOTSTRAP_REPEATS = 1000
RANDOM_STATE = 42
REUSE_CACHE = True

plt.style.use("seaborn-v0_8-whitegrid")
pd.set_option("display.max_columns", 160)
pd.set_option("display.width", 220)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")

def new_connection():
    """Open a new MatrixOne connection. / 创建新的 MatrixOne 连接。"""
    return pymysql.connect(
        host=MO_HOST,
        port=MO_PORT,
        user=MO_USER,
        password=MO_PASSWORD,
        database=MO_DB,
        charset="utf8mb4",
        autocommit=True,
        read_timeout=1800,
        write_timeout=1800,
        local_infile=True,
    )

conn = new_connection()

def query_df(sql, params=None, retry=True):
    """Run a SQL query and return a pandas DataFrame. / 执行 SQL 查询并返回 pandas DataFrame。"""
    global conn
    try:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            return pd.read_sql_query(sql, conn, params=params)
    except pymysql.err.OperationalError:
        if not retry:
            raise
        try:
            conn.close()
        except Exception:
            pass
        conn = new_connection()
        return query_df(sql, params=params, retry=False)

def table_exists(table_name):
    """Check whether a MatrixOne table exists. / 检查 MatrixOne 表是否存在。"""
    sql = '''
    SELECT COUNT(*) AS n
    FROM information_schema.tables
    WHERE table_schema = %s AND table_name = %s
    '''
    return int(query_df(sql, [MO_DB, table_name]).loc[0, "n"]) > 0

def sql_quote(value):
    """Escape a value for use in a SQL string literal. / 转义 SQL 字符串字面量。"""
    return "'" + str(value).replace("'", "''") + "'"

print("Connected to MatrixOne.")
print("Output directory:", OUTPUT_DIR)


In [ ]:
# Cell 3 - Validate H3 data and select Top 10 cells / 验证 H3 数据并选择 Top 10 单元

if not table_exists(ANALYSIS_TABLE):
    raise RuntimeError(
        f"Missing table: {ANALYSIS_TABLE}. "
        "Please run the H3 build notebook first."
    )

rows_h3 = query_df(
    f"SELECT COUNT(*) AS rows_h3 FROM {ANALYSIS_TABLE};"
)

source_split = query_df(f'''
SELECT source_file, COUNT(*) AS n
FROM {ANALYSIS_TABLE}
GROUP BY source_file
ORDER BY source_file;
''')

top_cells_df = query_df(f'''
SELECT pickup_h3 AS h3, COUNT(*) AS train_trip_count
FROM {ANALYSIS_TABLE}
WHERE COALESCE(shared_trip_authorized, 0) = 0
  AND pickup_h3 IS NOT NULL
  AND trip_start_timestamp >= '2022-01-01'
  AND trip_start_timestamp < '2024-01-01'
GROUP BY pickup_h3
ORDER BY train_trip_count DESC
LIMIT {TOP_N_CELLS};
''')

top_cells = top_cells_df["h3"].astype(str).tolist()

if len(top_cells) != TOP_N_CELLS:
    raise RuntimeError(
        f"Expected {TOP_N_CELLS} top cells, got {len(top_cells)}"
    )

display(rows_h3)
display(source_split)
display(top_cells_df)

rows_h3.to_csv(OUTPUT_DIR / "validation_rows_h3.csv", index=False)
source_split.to_csv(OUTPUT_DIR / "validation_source_split.csv", index=False)
top_cells_df.to_csv(
    OUTPUT_DIR / "top_h3_cells_train_2022_2023.csv",
    index=False,
)


In [ ]:
# Cell 4 - Load hourly pickup demand / 读取小时级上车需求

hourly_cache = (
    CACHE_DIR / f"hourly_pickup_top{TOP_N_CELLS}_2022_2024.csv.gz"
)
quoted_top_cells = ",".join(
    sql_quote(cell) for cell in top_cells
)

def query_hourly_by_year(year):
    """Load hourly pickup demand for one calendar year. / 读取一个自然年的小时级上车需求。"""
    return query_df(f'''
    SELECT
        pickup_h3 AS h3,
        DATE_FORMAT(
            trip_start_timestamp,
            '%Y-%m-%d %H:00:00'
        ) AS timestamp,
        COUNT(*) AS pickup_count
    FROM {ANALYSIS_TABLE}
    WHERE COALESCE(shared_trip_authorized, 0) = 0
      AND pickup_h3 IN ({quoted_top_cells})
      AND trip_start_timestamp >= '{year}-01-01'
      AND trip_start_timestamp < '{year + 1}-01-01'
    GROUP BY pickup_h3, timestamp;
    ''')

if REUSE_CACHE and hourly_cache.exists():
    print("Loading hourly demand cache:", hourly_cache)
    hourly_raw = pd.read_csv(
        hourly_cache,
        compression="gzip",
        dtype={"h3": str},
        parse_dates=["timestamp"],
    )
else:
    parts = []
    started = time.time()

    for year in [2022, 2023, 2024]:
        print("Querying MatrixOne year:", year)
        part = query_hourly_by_year(year)
        part["h3"] = part["h3"].astype(str)
        part["timestamp"] = pd.to_datetime(part["timestamp"])
        parts.append(part)

    hourly_raw = pd.concat(parts, ignore_index=True)
    hourly_raw.to_csv(
        hourly_cache,
        index=False,
        compression="gzip",
    )

    print(
        "MatrixOne query minutes:",
        round((time.time() - started) / 60, 2),
    )

print("Non-empty H3-hour rows:", f"{len(hourly_raw):,}")
display(hourly_raw.head())


In [ ]:
# Cell 5 - Complete the H3-hour grid and calendar features / 补全 H3 小时网格与日历特征

all_hours = pd.date_range(DATA_START, DATA_END, freq="h")

hourly_grid = pd.MultiIndex.from_product(
    [top_cells, all_hours],
    names=["h3", "timestamp"],
).to_frame(index=False)

hourly = hourly_grid.merge(
    hourly_raw,
    on=["h3", "timestamp"],
    how="left",
)

hourly["pickup_count"] = (
    hourly["pickup_count"].fillna(0).astype("int32")
)

hourly = hourly.sort_values(
    ["h3", "timestamp"]
).reset_index(drop=True)

hourly["year"] = hourly["timestamp"].dt.year.astype("int16")
hourly["month"] = hourly["timestamp"].dt.month.astype("int8")
hourly["hour"] = hourly["timestamp"].dt.hour.astype("int8")
hourly["dow_mon0"] = hourly["timestamp"].dt.dayofweek.astype("int8")
hourly["day_of_year"] = hourly["timestamp"].dt.dayofyear.astype("int16")
hourly["is_weekend"] = (hourly["dow_mon0"] >= 5).astype("int8")

us_holidays = holidays.US(years=[2022, 2023, 2024])
hourly["is_holiday"] = (
    hourly["timestamp"].dt.date
    .map(lambda x: int(x in us_holidays))
    .astype("int8")
)

def nth_weekday(year, month, weekday, n):
    """Return the nth requested weekday in a month. / 返回某月第 n 个指定星期日期。"""
    first = date(year, month, 1)
    shift = (weekday - first.weekday()) % 7
    return first + timedelta(days=shift + 7 * (n - 1))

dst_start_dates = {
    nth_weekday(year, 3, calendar.SUNDAY, 2)
    for year in [2022, 2023, 2024]
}

hourly["is_dst_missing_hour"] = (
    hourly["timestamp"].dt.date.isin(dst_start_dates)
    & (hourly["hour"] == 2)
).astype("int8")

hourly["hour_sin"] = np.sin(2 * np.pi * hourly["hour"] / 24)
hourly["hour_cos"] = np.cos(2 * np.pi * hourly["hour"] / 24)
hourly["dow_sin"] = np.sin(2 * np.pi * hourly["dow_mon0"] / 7)
hourly["dow_cos"] = np.cos(2 * np.pi * hourly["dow_mon0"] / 7)
hourly["doy_sin"] = np.sin(
    2 * np.pi * hourly["day_of_year"] / 365.25
)
hourly["doy_cos"] = np.cos(
    2 * np.pi * hourly["day_of_year"] / 365.25
)

print("Complete H3-hour rows:", f"{len(hourly):,}")
print(
    "Zero pickup-hour ratio:",
    round((hourly["pickup_count"] == 0).mean() * 100, 3),
    "%",
)
display(hourly.head())


In [ ]:
# Cell 6 - Load Chicago historical weather / 读取芝加哥历史天气

WEATHER_CACHE = (
    CACHE_DIR / "open_meteo_archive_era5_chicago_2022_2024.csv.gz"
)
OPEN_METEO_URL = "https://archive-api.open-meteo.com/v1/archive"

WEATHER_VARIABLES = [
    "temperature_2m",
    "apparent_temperature",
    "relative_humidity_2m",
    "precipitation",
    "rain",
    "snowfall",
    "cloud_cover",
    "wind_speed_10m",
    "weather_code",
]

def standardize_weather(df):
    """Standardize weather columns and timestamps. / 统一天气字段和时间格式。"""
    df = df.copy()

    if "time" in df.columns and "timestamp" not in df.columns:
        df = df.rename(columns={"time": "timestamp"})

    df["timestamp"] = pd.to_datetime(
        df["timestamp"],
        errors="coerce",
    )
    df = df.dropna(subset=["timestamp"])

    for column in WEATHER_VARIABLES:
        if column not in df.columns:
            df[column] = np.nan
        df[column] = pd.to_numeric(
            df[column],
            errors="coerce",
        )

    aggregation = {
        "temperature_2m": "mean",
        "apparent_temperature": "mean",
        "relative_humidity_2m": "mean",
        "precipitation": "sum",
        "rain": "sum",
        "snowfall": "sum",
        "cloud_cover": "mean",
        "wind_speed_10m": "mean",
        "weather_code": "max",
    }

    df = (
        df[["timestamp"] + WEATHER_VARIABLES]
        .groupby("timestamp", as_index=False)
        .agg(aggregation)
        .sort_values("timestamp")
    )

    complete_index = pd.date_range(
        DATA_START,
        DATA_END,
        freq="h",
    )

    df = (
        df.set_index("timestamp")
        .reindex(complete_index)
        .rename_axis("timestamp")
        .reset_index()
    )

    df["weather_imputed"] = (
        df[WEATHER_VARIABLES].isna().any(axis=1).astype("int8")
    )

    continuous = [
        "temperature_2m",
        "apparent_temperature",
        "relative_humidity_2m",
        "cloud_cover",
        "wind_speed_10m",
    ]

    df[continuous] = (
        df[continuous]
        .interpolate(limit=3, limit_direction="both")
    )

    df[["precipitation", "rain", "snowfall"]] = (
        df[["precipitation", "rain", "snowfall"]]
        .fillna(0.0)
    )

    df["weather_code"] = (
        df["weather_code"].ffill().bfill()
    )

    return df

def download_weather():
    """Download or load cached Chicago weather observations. / 下载或读取缓存的芝加哥天气观测。"""
    params = {
        "latitude": 41.8781,
        "longitude": -87.6298,
        "start_date": DATA_START.strftime("%Y-%m-%d"),
        "end_date": DATA_END.strftime("%Y-%m-%d"),
        "hourly": ",".join(WEATHER_VARIABLES),
        "timezone": "America/Chicago",
        "temperature_unit": "celsius",
        "wind_speed_unit": "kmh",
        "precipitation_unit": "mm",
        "models": "era5",
    }

    response = requests.get(
        OPEN_METEO_URL,
        params=params,
        timeout=180,
        verify=certifi.where(),
        headers={"User-Agent": "Chicago-TNP-study/4.0"},
    )

    if response.status_code == 400:
        params.pop("models", None)
        response = requests.get(
            OPEN_METEO_URL,
            params=params,
            timeout=180,
            verify=certifi.where(),
            headers={"User-Agent": "Chicago-TNP-study/4.0"},
        )

    response.raise_for_status()
    payload = response.json()

    if "hourly" not in payload:
        raise RuntimeError("Weather response has no hourly data.")

    return pd.DataFrame(payload["hourly"])

if REUSE_CACHE and WEATHER_CACHE.exists():
    print("Loading weather cache:", WEATHER_CACHE)
    weather = pd.read_csv(
        WEATHER_CACHE,
        compression="gzip",
        parse_dates=["timestamp"],
    )
    weather_source = "cached_open_meteo_archive"
else:
    weather = standardize_weather(download_weather())
    weather.to_csv(
        WEATHER_CACHE,
        index=False,
        compression="gzip",
    )
    weather_source = "open_meteo_archive_era5"

weather = standardize_weather(weather)

print("Weather source:", weather_source)
print("Weather rows:", len(weather))
print("Imputed weather hours:", int(weather["weather_imputed"].sum()))
display(weather.head())


In [ ]:
# Cell 7 - Merge weather and prepare the model table / 合并天气并准备建模表

weather["rain_flag"] = (weather["rain"] > 0).astype("int8")
weather["heavy_rain_flag"] = (weather["rain"] >= 2.5).astype("int8")
weather["snow_flag"] = (weather["snowfall"] > 0).astype("int8")
weather["high_wind_flag"] = (
    weather["wind_speed_10m"] >= 35
).astype("int8")

weather["weather_type"] = np.select(
    [
        weather["snow_flag"] == 1,
        weather["rain_flag"] == 1,
        weather["high_wind_flag"] == 1,
    ],
    [
        "snow",
        "rain_no_snow",
        "high_wind_only",
    ],
    default="dry",
)

model_df = hourly.merge(
    weather,
    on="timestamp",
    how="left",
    validate="many_to_one",
)

critical_weather = [
    "temperature_2m",
    "rain",
    "snowfall",
    "wind_speed_10m",
]

if model_df[critical_weather].isna().any().any():
    raise RuntimeError("Weather merge produced missing values.")

baseline = (
    model_df[model_df["timestamp"] < TRAIN_END]
    .groupby(["h3", "dow_mon0", "hour"], as_index=False)
    .agg(
        expected_pickup=("pickup_count", "mean"),
        std_pickup=("pickup_count", "std"),
    )
)

baseline["std_pickup"] = (
    baseline["std_pickup"]
    .replace(0, np.nan)
    .fillna(1.0)
    .clip(lower=1.0)
)

model_df = model_df.merge(
    baseline,
    on=["h3", "dow_mon0", "hour"],
    how="left",
    validate="many_to_one",
)

model_df["pickup_z"] = (
    model_df["pickup_count"]
    - model_df["expected_pickup"]
) / model_df["std_pickup"]

model_df["is_high_peak"] = (
    (model_df["pickup_z"] >= HIGH_Z_THRESHOLD)
    & (model_df["pickup_count"] >= MIN_PEAK_COUNT)
    & (model_df["expected_pickup"] >= MIN_EXPECTED_FOR_PEAK)
    & (model_df["is_dst_missing_hour"] == 0)
).astype("int8")

model_df = model_df.rename(
    columns={"pickup_count": "target"}
)

model_df["id"] = model_df["h3"].astype(str)

model_df["seasonal_naive_168h"] = (
    model_df.groupby("id", sort=False)["target"].shift(168)
)

CALENDAR_COVARIATES = [
    "hour_sin",
    "hour_cos",
    "dow_sin",
    "dow_cos",
    "doy_sin",
    "doy_cos",
    "is_weekend",
    "is_holiday",
]

WEATHER_COVARIATES = [
    "temperature_2m",
    "apparent_temperature",
    "relative_humidity_2m",
    "precipitation",
    "rain",
    "snowfall",
    "cloud_cover",
    "wind_speed_10m",
    "weather_code",
    "rain_flag",
    "heavy_rain_flag",
    "snow_flag",
    "high_wind_flag",
]

for column in CALENDAR_COVARIATES + WEATHER_COVARIATES:
    model_df[column] = pd.to_numeric(
        model_df[column],
        errors="coerce",
    )

if model_df[CALENDAR_COVARIATES + WEATHER_COVARIATES].isna().any().any():
    raise RuntimeError("Chronos covariates contain missing values.")

print("Prepared model rows:", f"{len(model_df):,}")
display(model_df.head())


In [ ]:
# Cell 8 - Create balanced 2024 backtest origins / 构造平衡的 2024 回测起点

first_origin = max(
    TRAIN_END,
    DATA_START + pd.Timedelta(hours=CONTEXT_LENGTH),
)

last_origin = (
    DATA_END
    - pd.Timedelta(hours=PREDICTION_LENGTH - 1)
)

backtest_origins = pd.date_range(
    first_origin,
    last_origin,
    freq=f"{BACKTEST_STRIDE_HOURS}h",
)

origin_info = pd.DataFrame(
    {"forecast_origin": backtest_origins}
)
origin_info["hour"] = origin_info["forecast_origin"].dt.hour
origin_info["dow"] = origin_info["forecast_origin"].dt.dayofweek

print("Backtest origins:", len(backtest_origins))
print("Unique origin hours:", origin_info["hour"].nunique())
print("Unique weekdays:", origin_info["dow"].nunique())

display(
    origin_info["hour"]
    .value_counts()
    .sort_index()
    .rename_axis("hour")
    .reset_index(name="n")
)

display(
    origin_info["dow"]
    .value_counts()
    .sort_index()
    .rename_axis("dow")
    .reset_index(name="n")
)

origin_info.to_csv(
    OUTPUT_DIR / "balanced_backtest_origins.csv",
    index=False,
)


# 隔离 Chronos 子进程

接下来的 Cell 9 会把已经准备好的小时级数据保存成 Parquet，并生成一个独立的 `chronos_worker.py`。

这个 worker 才会导入：

```python
torch
Chronos2Pipeline
```

Jupyter 主进程不会导入它们。


In [ ]:
# Cell 9 - Export data and worker configuration / 导出数据与进程配置

WORKER_DATA_PATH = WORKER_DIR / "chronos_model_data.parquet"
WORKER_CONFIG_PATH = WORKER_DIR / "chronos_config.json"
WORKER_SCRIPT_PATH = WORKER_DIR / "chronos_worker.py"

worker_columns = [
    "id",
    "h3",
    "timestamp",
    "target",
    "seasonal_naive_168h",
    "expected_pickup",
    "std_pickup",
    "pickup_z",
    "is_high_peak",
    "is_dst_missing_hour",
    "weather_type",
] + CALENDAR_COVARIATES + WEATHER_COVARIATES

model_df[worker_columns].to_parquet(
    WORKER_DATA_PATH,
    index=False,
)

worker_config = {
    "data_path": str(WORKER_DATA_PATH),
    "output_dir": str(WORKER_DIR),
    "model_id": CHRONOS_MODEL_ID,
    "device": CHRONOS_DEVICE,
    "threads": CHRONOS_THREADS,
    "context_length": CONTEXT_LENGTH,
    "prediction_length": PREDICTION_LENGTH,
    "top_n_cells": TOP_N_CELLS,
    "calendar_covariates": CALENDAR_COVARIATES,
    "weather_covariates": WEATHER_COVARIATES,
    "origins": [
        origin.isoformat()
        for origin in backtest_origins
    ],
}

with open(
    WORKER_CONFIG_PATH,
    "w",
    encoding="utf-8",
) as file:
    json.dump(worker_config, file, ensure_ascii=False, indent=2)

print("Worker data:", WORKER_DATA_PATH)
print("Worker config:", WORKER_CONFIG_PATH)
print("Rows exported:", f"{len(model_df):,}")


In [ ]:
# Cell 10 - Write the isolated Chronos worker / 写入隔离的 Chronos 进程

worker_code = r"""
import os
import sys
import gc
import json
import time
import argparse
from pathlib import Path

os.environ.setdefault("OMP_NUM_THREADS", "1")
os.environ.setdefault("VECLIB_MAXIMUM_THREADS", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

import numpy as np
import pandas as pd
import torch
from chronos import Chronos2Pipeline


def normalize_prediction(pred_df, prefix, quantile_levels):
    '''Normalize Chronos predictions to one shared schema. / 将 Chronos 预测统一到同一数据结构。'''
    pred_df = pred_df.copy()
    pred_df["id"] = pred_df["id"].astype(str)
    pred_df["timestamp"] = pd.to_datetime(pred_df["timestamp"])

    if "predictions" in pred_df.columns:
        pred_df = pred_df.rename(
            columns={"predictions": f"{prefix}_median"}
        )

    for quantile in quantile_levels:
        for candidate in [
            quantile,
            str(quantile),
            f"{quantile:.1f}",
        ]:
            if candidate in pred_df.columns:
                pred_df = pred_df.rename(
                    columns={
                        candidate:
                        f"{prefix}_q{int(quantile * 100):02d}"
                    }
                )
                break

    if (
        f"{prefix}_median" not in pred_df.columns
        and f"{prefix}_q50" in pred_df.columns
    ):
        pred_df[f"{prefix}_median"] = pred_df[f"{prefix}_q50"]

    if f"{prefix}_median" not in pred_df.columns:
        raise RuntimeError(
            f"Unexpected Chronos columns: {pred_df.columns.tolist()}"
        )

    prediction_columns = [
        column
        for column in pred_df.columns
        if column.startswith(prefix + "_")
    ]

    for column in prediction_columns:
        pred_df[column] = (
            pd.to_numeric(pred_df[column], errors="coerce")
            .clip(lower=0)
        )

    return pred_df[
        ["id", "timestamp"] + prediction_columns
    ]


def predict_no_covariates(
    pipeline,
    context_df,
    prediction_length,
    quantile_levels,
):
    '''Run Chronos without future covariates. / 运行不使用未来协变量的 Chronos 预测。'''
    with torch.inference_mode():
        pred = pipeline.predict_df(
            context_df[["id", "timestamp", "target"]],
            prediction_length=prediction_length,
            quantile_levels=quantile_levels,
            id_column="id",
            timestamp_column="timestamp",
            target="target",
        )

    return normalize_prediction(
        pred,
        "chronos_no_covariates",
        quantile_levels,
    )


def predict_calendar(
    pipeline,
    context_df,
    future_df,
    prediction_length,
    quantile_levels,
    calendar_covariates,
):
    '''Run Chronos with known calendar covariates. / 运行使用已知日历协变量的 Chronos 预测。'''
    with torch.inference_mode():
        pred = pipeline.predict_df(
            context_df[
                ["id", "timestamp", "target"]
                + calendar_covariates
            ],
            future_df=future_df[
                ["id", "timestamp"]
                + calendar_covariates
            ],
            prediction_length=prediction_length,
            quantile_levels=quantile_levels,
            id_column="id",
            timestamp_column="timestamp",
            target="target",
        )

    return normalize_prediction(
        pred,
        "chronos_calendar",
        quantile_levels,
    )


def predict_weather(
    pipeline,
    context_df,
    future_df,
    prediction_length,
    quantile_levels,
    calendar_covariates,
    weather_covariates,
):
    '''Run Chronos with calendar and weather covariates. / 运行使用日历和天气协变量的 Chronos 预测。'''
    covariates = calendar_covariates + weather_covariates

    with torch.inference_mode():
        pred = pipeline.predict_df(
            context_df[
                ["id", "timestamp", "target"]
                + covariates
            ],
            future_df=future_df[
                ["id", "timestamp"]
                + covariates
            ],
            prediction_length=prediction_length,
            quantile_levels=quantile_levels,
            id_column="id",
            timestamp_column="timestamp",
            target="target",
        )

    return normalize_prediction(
        pred,
        "chronos_calendar_weather",
        quantile_levels,
    )


def main():
    '''Execute the isolated Chronos forecasting worker. / 执行隔离的 Chronos 预测进程。'''
    parser = argparse.ArgumentParser()
    parser.add_argument("--config", required=True)
    parser.add_argument("--run-name", required=True)
    parser.add_argument("--max-origins", type=int, default=None)
    args = parser.parse_args()

    with open(args.config, "r", encoding="utf-8") as file:
        config = json.load(file)

    threads = int(config.get("threads", 1))
    torch.set_num_threads(max(1, threads))

    data_path = Path(config["data_path"])
    output_dir = Path(config["output_dir"])
    run_dir = output_dir / f"{args.run_name}_chunks"
    run_dir.mkdir(parents=True, exist_ok=True)

    final_path = output_dir / f"{args.run_name}_predictions.csv.gz"

    data = pd.read_parquet(data_path)
    data["timestamp"] = pd.to_datetime(data["timestamp"])
    data["id"] = data["id"].astype(str)

    origins = [
        pd.Timestamp(value)
        for value in config["origins"]
    ]

    if args.max_origins is not None:
        origins = origins[:args.max_origins]

    context_length = int(config["context_length"])
    prediction_length = int(config["prediction_length"])
    top_n_cells = int(config["top_n_cells"])
    calendar_covariates = list(config["calendar_covariates"])
    weather_covariates = list(config["weather_covariates"])
    quantile_levels = [0.1, 0.5, 0.9]

    print("Worker PID:", os.getpid(), flush=True)
    print("Python:", sys.version, flush=True)
    print("Torch:", torch.__version__, flush=True)
    print("Device:", config["device"], flush=True)
    print("Origins:", len(origins), flush=True)
    print("Loading model...", flush=True)

    try:
        pipeline = Chronos2Pipeline.from_pretrained(
            config["model_id"],
            device_map=config["device"],
            attn_implementation="eager",
        )
    except TypeError:
        pipeline = Chronos2Pipeline.from_pretrained(
            config["model_id"],
            device_map=config["device"],
        )

    print("Model loaded.", flush=True)

    completed_paths = []
    started = time.time()

    for index, origin in enumerate(origins, 1):
        key = origin.strftime("%Y%m%d_%H%M")
        chunk_path = run_dir / f"origin_{key}.parquet"

        if chunk_path.exists():
            completed_paths.append(chunk_path)
            print(
                f"{index}/{len(origins)} reused {origin}",
                flush=True,
            )
            continue

        context_start = origin - pd.Timedelta(
            hours=context_length
        )
        forecast_end = origin + pd.Timedelta(
            hours=prediction_length
        )

        context_df = data[
            (data["timestamp"] >= context_start)
            & (data["timestamp"] < origin)
        ].copy()

        future_df = data[
            (data["timestamp"] >= origin)
            & (data["timestamp"] < forecast_end)
        ].copy()

        expected_context_rows = top_n_cells * context_length
        expected_future_rows = top_n_cells * prediction_length

        if len(context_df) != expected_context_rows:
            raise RuntimeError(
                f"Context mismatch at {origin}: "
                f"{len(context_df)} != {expected_context_rows}"
            )

        if len(future_df) != expected_future_rows:
            raise RuntimeError(
                f"Future mismatch at {origin}: "
                f"{len(future_df)} != {expected_future_rows}"
            )

        context_df = context_df.sort_values(
            ["id", "timestamp"]
        )
        future_df = future_df.sort_values(
            ["id", "timestamp"]
        )

        keep_columns = [
            "id",
            "h3",
            "timestamp",
            "target",
            "seasonal_naive_168h",
            "expected_pickup",
            "std_pickup",
            "pickup_z",
            "is_high_peak",
            "is_dst_missing_hour",
            "weather_type",
        ] + calendar_covariates + weather_covariates

        merged = future_df[keep_columns].copy()

        pred = predict_no_covariates(
            pipeline,
            context_df,
            prediction_length,
            quantile_levels,
        )
        merged = merged.merge(
            pred,
            on=["id", "timestamp"],
            how="left",
            validate="one_to_one",
        )
        del pred
        gc.collect()

        pred = predict_calendar(
            pipeline,
            context_df,
            future_df,
            prediction_length,
            quantile_levels,
            calendar_covariates,
        )
        merged = merged.merge(
            pred,
            on=["id", "timestamp"],
            how="left",
            validate="one_to_one",
        )
        del pred
        gc.collect()

        pred = predict_weather(
            pipeline,
            context_df,
            future_df,
            prediction_length,
            quantile_levels,
            calendar_covariates,
            weather_covariates,
        )
        merged = merged.merge(
            pred,
            on=["id", "timestamp"],
            how="left",
            validate="one_to_one",
        )
        del pred
        gc.collect()

        merged["forecast_origin"] = origin
        merged["horizon"] = (
            (
                merged["timestamp"] - origin
            ) / pd.Timedelta(hours=1)
        ).astype(int) + 1

        merged.to_parquet(chunk_path, index=False)
        completed_paths.append(chunk_path)

        del context_df
        del future_df
        del merged
        gc.collect()

        elapsed = (time.time() - started) / 60

        print(
            f"{index}/{len(origins)} completed {origin}; "
            f"elapsed={elapsed:.2f} min",
            flush=True,
        )

    if not completed_paths:
        raise RuntimeError("No Chronos chunks were produced.")

    combined = pd.concat(
        [
            pd.read_parquet(path)
            for path in sorted(completed_paths)
        ],
        ignore_index=True,
    )

    combined = combined.sort_values(
        ["forecast_origin", "id", "timestamp"]
    ).reset_index(drop=True)

    combined.to_csv(
        final_path,
        index=False,
        compression="gzip",
    )

    print("Final predictions:", final_path, flush=True)
    print("Rows:", len(combined), flush=True)
    print("CHRONOS_WORKER_SUCCESS", flush=True)


if __name__ == "__main__":
    main()
"""

WORKER_SCRIPT_PATH.write_text(
    worker_code,
    encoding="utf-8",
)

print("Worker script written:", WORKER_SCRIPT_PATH)
print("Worker script size:", WORKER_SCRIPT_PATH.stat().st_size, "bytes")


In [ ]:
import os
# Cell 11 - Run the Chronos worker and stream output / 运行 Chronos 进程并输出日志

def run_chronos_worker(run_name, max_origins=None):
    """Run the isolated Chronos process and stream its logs. / 运行隔离的 Chronos 进程并输出日志。"""
    command = [
        sys.executable,
        str(WORKER_SCRIPT_PATH),
        "--config",
        str(WORKER_CONFIG_PATH),
        "--run-name",
        run_name,
    ]

    if max_origins is not None:
        command.extend(
            ["--max-origins", str(max_origins)]
        )

    worker_env = os.environ.copy()
    worker_env.update({
        "OMP_NUM_THREADS": "1",
        "VECLIB_MAXIMUM_THREADS": "1",
        "TOKENIZERS_PARALLELISM": "false",
        "PYTORCH_ENABLE_MPS_FALLBACK": "1",
    })

    print("Starting isolated worker:")
    print(" ".join(command))
    print("Jupyter PID remains:", os.getpid())

    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
        env=worker_env,
    )

    assert process.stdout is not None

    for line in process.stdout:
        print(line, end="")

    return_code = process.wait()

    print("Worker return code:", return_code)

    if return_code != 0:
        raise RuntimeError(
            "Chronos worker failed, but the Jupyter kernel survived. "
            f"Return code: {return_code}. "
            "A negative return code such as -11 indicates SIGSEGV."
        )

    return WORKER_DIR / f"{run_name}_predictions.csv.gz"


## 12. One-origin validation run / 单起点验证运行

This validation executes one forecast origin through the isolated Chronos worker and verifies the process boundary, output schema, and checkpoint path before the full backtest.  
该验证通过隔离的 Chronos 进程执行一个预测起点，在完整回测前检查进程边界、输出结构和检查点路径。


In [ ]:
# Cell 12 - One-origin validation run / 单起点验证运行

SMOKE_OUTPUT_PATH = run_chronos_worker(
    run_name="chronos_smoke",
    max_origins=1,
)

smoke_predictions = pd.read_csv(
    SMOKE_OUTPUT_PATH,
    compression="gzip",
    parse_dates=["timestamp", "forecast_origin"],
    dtype={"id": str, "h3": str},
)

print("Smoke rows:", len(smoke_predictions))
display(smoke_predictions.head())


## 13. Full Chronos backtest / 完整 Chronos 回测

The full backtest evaluates all selected 2024 forecast origins. Each completed origin is stored immediately, allowing deterministic resume after interruption.  
完整回测评估全部选定的 2024 预测起点。每个完成的起点会立即保存，因此中断后可以确定性续跑。


In [ ]:
# Cell 13 - Full isolated Chronos backtest / 完整隔离式 Chronos 回测

CHRONOS_OUTPUT_PATH = run_chronos_worker(
    run_name="chronos_full",
    max_origins=None,
)

chronos_predictions = pd.read_csv(
    CHRONOS_OUTPUT_PATH,
    compression="gzip",
    parse_dates=["timestamp", "forecast_origin"],
    dtype={"id": str, "h3": str},
)

print("Chronos full rows:", f"{len(chronos_predictions):,}")
display(chronos_predictions.head())


# LightGBM机器学习模型

Chronos子进程已经结束后，主Notebook才导入LightGBM。

因此 PyTorch/Chronos 与 LightGBM 永远不会进入同一个 Python 进程。


In [ ]:
# Cell 14 - Build leakage-safe LightGBM features / 构造无泄漏 LightGBM 特征

from lightgbm import LGBMRegressor

model_df = model_df.sort_values(
    ["id", "timestamp"]
).reset_index(drop=True)

id_categories = pd.Categorical(
    model_df["id"],
    categories=top_cells,
)
model_df["id_code"] = id_categories.codes.astype("int16")

grouped_target = model_df.groupby(
    "id",
    sort=False,
)["target"]

ML_LAG_HOURS = [1, 2, 3, 6, 12, 24, 48, 168]

for lag in ML_LAG_HOURS:
    model_df[f"lag_{lag}h"] = grouped_target.shift(lag)

for window in [6, 24, 168]:
    model_df[f"roll_mean_{window}h"] = (
        model_df.groupby(
            "id",
            sort=False,
        )["target"]
        .transform(
            lambda series: (
                series.shift(1)
                .rolling(
                    window,
                    min_periods=max(2, window // 3),
                )
                .mean()
            )
        )
    )

for window in [24, 168]:
    model_df[f"roll_std_{window}h"] = (
        model_df.groupby(
            "id",
            sort=False,
        )["target"]
        .transform(
            lambda series: (
                series.shift(1)
                .rolling(
                    window,
                    min_periods=max(2, window // 3),
                )
                .std()
            )
        )
    )

model_df["trend_1h"] = (
    model_df["lag_1h"] - model_df["lag_2h"]
)
model_df["trend_6h"] = (
    model_df["lag_1h"] - model_df["lag_6h"]
)
model_df["trend_24h"] = (
    model_df["lag_1h"] - model_df["lag_24h"]
)

ML_HISTORY_FEATURES = (
    [f"lag_{lag}h" for lag in ML_LAG_HOURS]
    + [
        "roll_mean_6h",
        "roll_mean_24h",
        "roll_mean_168h",
        "roll_std_24h",
        "roll_std_168h",
        "trend_1h",
        "trend_6h",
        "trend_24h",
    ]
)

FUTURE_CALENDAR_FEATURES = [
    f"future_{column}"
    for column in CALENDAR_COVARIATES
]

FUTURE_WEATHER_FEATURES = [
    f"future_{column}"
    for column in WEATHER_COVARIATES
]

ML_COMMON_FEATURES = (
    ["id_code", "horizon"]
    + ML_HISTORY_FEATURES
    + ["future_expected_pickup", "future_std_pickup"]
    + FUTURE_CALENDAR_FEATURES
)

ML_WEATHER_FEATURES = (
    ML_COMMON_FEATURES
    + FUTURE_WEATHER_FEATURES
)

print("LightGBM feature preparation completed.")


In [ ]:
# Cell 15 - Build time-aligned LightGBM datasets / 构造时间对齐的 LightGBM 数据集
# Time alignment / 时间对齐：horizon=1 必须对应 forecast_origin 本身，
# 与 Chronos 的第一个预测小时保持完全一致。

ML_DATA_CACHE = (
    CACHE_DIR / "lightgbm_multihorizon_data_time_aligned.pkl"
)

origin_columns = (
    ["id", "id_code", "timestamp"]
    + ML_HISTORY_FEATURES
)

future_columns = (
    [
        "id",
        "timestamp",
        "target",
        "expected_pickup",
        "std_pickup",
    ]
    + CALENDAR_COVARIATES
    + WEATHER_COVARIATES
)

future_lookup = model_df[future_columns].copy()

rename_map = {
    "timestamp": "target_timestamp",
    "target": "label",
    "expected_pickup": "future_expected_pickup",
    "std_pickup": "future_std_pickup",
}

for column in CALENDAR_COVARIATES:
    rename_map[column] = f"future_{column}"

for column in WEATHER_COVARIATES:
    rename_map[column] = f"future_{column}"

future_lookup = future_lookup.rename(
    columns=rename_map
)

def make_ml_dataset(origin_times):
    """Build time-aligned LightGBM training or test rows. / 构造时间对齐的 LightGBM 训练或测试样本。"""
    origins = (
        model_df[
            model_df["timestamp"].isin(origin_times)
        ][origin_columns]
        .rename(columns={"timestamp": "forecast_origin"})
        .copy()
    )

    horizons = pd.DataFrame({
        "horizon": np.arange(
            1,
            PREDICTION_LENGTH + 1,
        )
    })

    expanded = origins.merge(
        horizons,
        how="cross",
    )

    # Chronos 定义：
    # horizon=1 -> forecast_origin
    # horizon=24 -> forecast_origin + 23 hours
    expanded["target_timestamp"] = (
        expanded["forecast_origin"]
        + pd.to_timedelta(
            expanded["horizon"] - 1,
            unit="h",
        )
    )

    expanded = expanded.merge(
        future_lookup,
        on=["id", "target_timestamp"],
        how="left",
        validate="many_to_one",
    )

    expanded = expanded.dropna(
        subset=["label"] + ML_HISTORY_FEATURES
    )

    expanded["id_code"] = (
        expanded["id_code"]
        .astype("int16")
        .astype("category")
    )

    return expanded

if REUSE_CACHE and ML_DATA_CACHE.exists():
    print("Loading corrected LightGBM cache:", ML_DATA_CACHE)
    cached = joblib.load(ML_DATA_CACHE)
    ml_train = cached["train"]
    ml_test = cached["test"]
else:
    earliest_origin = (
        DATA_START
        + pd.Timedelta(hours=max(ML_LAG_HOURS))
    )

    latest_train_origin = (
        TRAIN_END
        - pd.Timedelta(hours=PREDICTION_LENGTH - 1)
    )

    train_candidates = (
        model_df[
            (model_df["timestamp"] >= earliest_origin)
            & (model_df["timestamp"] < latest_train_origin)
        ]["timestamp"]
        .drop_duplicates()
        .sort_values()
    )

    offsets = (
        (train_candidates - earliest_origin)
        / pd.Timedelta(hours=1)
    ).astype(int)

    train_origins = train_candidates[
        offsets % ML_TRAIN_ORIGIN_STRIDE_HOURS == 0
    ]

    ml_train = make_ml_dataset(train_origins)
    ml_test = make_ml_dataset(backtest_origins)

    joblib.dump(
        {
            "train": ml_train,
            "test": ml_test,
        },
        ML_DATA_CACHE,
        compress=3,
    )

expected_test_rows = (
    len(backtest_origins)
    * TOP_N_CELLS
    * PREDICTION_LENGTH
)

print("LightGBM train rows:", f"{len(ml_train):,}")
print("LightGBM test rows:", f"{len(ml_test):,}")
print("Expected test rows:", f"{expected_test_rows:,}")

if len(ml_test) != expected_test_rows:
    raise RuntimeError(
        "Corrected LightGBM test rows still do not match "
        f"the Chronos design: {len(ml_test)} != {expected_test_rows}"
    )

alignment_check = ml_test[
    ["forecast_origin", "target_timestamp", "horizon"]
].head(24).copy()

alignment_check["expected_timestamp"] = (
    alignment_check["forecast_origin"]
    + pd.to_timedelta(
        alignment_check["horizon"] - 1,
        unit="h",
    )
)

if not (
    alignment_check["target_timestamp"]
    == alignment_check["expected_timestamp"]
).all():
    raise RuntimeError("LightGBM horizon alignment check failed.")

display(alignment_check.head())
print(
    "Time alignment passed: horizon 1 equals forecast origin; "
    "horizon 24 equals origin + 23 hours."
)


In [ ]:
# Cell 16 - Train and predict with LightGBM / 训练并运行 LightGBM 预测

def make_lightgbm_model():
    """Create the LightGBM forecasting model. / 创建 LightGBM 预测模型。"""
    return LGBMRegressor(
        objective="regression_l1",
        n_estimators=450,
        learning_rate=0.05,
        num_leaves=63,
        min_child_samples=100,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_alpha=0.1,
        reg_lambda=1.0,
        random_state=RANDOM_STATE,
        n_jobs=4,
        verbosity=-1,
    )

CALENDAR_MODEL_PATH = (
    MODEL_DIR / "lightgbm_calendar_time_aligned.joblib"
)
WEATHER_MODEL_PATH = (
    MODEL_DIR / "lightgbm_calendar_weather_time_aligned.joblib"
)

if (
    REUSE_CACHE
    and CALENDAR_MODEL_PATH.exists()
    and WEATHER_MODEL_PATH.exists()
):
    print("Loading cached LightGBM models.")
    lightgbm_calendar = joblib.load(
        CALENDAR_MODEL_PATH
    )
    lightgbm_weather = joblib.load(
        WEATHER_MODEL_PATH
    )
else:
    started = time.time()

    lightgbm_calendar = make_lightgbm_model()
    lightgbm_calendar.fit(
        ml_train[ML_COMMON_FEATURES],
        ml_train["label"],
        categorical_feature=["id_code"],
    )

    print(
        "Calendar model minutes:",
        round((time.time() - started) / 60, 2),
    )

    weather_started = time.time()

    lightgbm_weather = make_lightgbm_model()
    lightgbm_weather.fit(
        ml_train[ML_WEATHER_FEATURES],
        ml_train["label"],
        categorical_feature=["id_code"],
    )

    print(
        "Weather model minutes:",
        round((time.time() - weather_started) / 60, 2),
    )

    joblib.dump(
        lightgbm_calendar,
        CALENDAR_MODEL_PATH,
        compress=3,
    )
    joblib.dump(
        lightgbm_weather,
        WEATHER_MODEL_PATH,
        compress=3,
    )

ml_test = ml_test.copy()

ml_test["lightgbm_calendar"] = np.clip(
    lightgbm_calendar.predict(
        ml_test[ML_COMMON_FEATURES]
    ),
    0,
    None,
)

ml_test["lightgbm_calendar_weather"] = np.clip(
    lightgbm_weather.predict(
        ml_test[ML_WEATHER_FEATURES]
    ),
    0,
    None,
)

print("LightGBM predictions completed.")


In [ ]:
# Cell 17 - Merge aligned model predictions / 合并时间对齐的模型预测
# Validate after alignment / 对齐后验证：每一行都匹配，防止空模型进入评价表。

ml_predictions = ml_test[
    [
        "id",
        "forecast_origin",
        "target_timestamp",
        "horizon",
        "label",
        "lightgbm_calendar",
        "lightgbm_calendar_weather",
    ]
].rename(
    columns={
        "target_timestamp": "timestamp",
        "label": "ml_label_check",
    }
)

all_predictions = chronos_predictions.merge(
    ml_predictions,
    on=[
        "id",
        "timestamp",
        "forecast_origin",
        "horizon",
    ],
    how="left",
    validate="one_to_one",
)

label_mismatch = (
    all_predictions["target"]
    - all_predictions["ml_label_check"]
).abs().max()

if pd.notna(label_mismatch) and label_mismatch != 0:
    raise RuntimeError(
        "Chronos and LightGBM labels differ: "
        f"{label_mismatch}"
    )

required_ml_columns = [
    "ml_label_check",
    "lightgbm_calendar",
    "lightgbm_calendar_weather",
]

missing_counts = (
    all_predictions[required_ml_columns]
    .isna()
    .sum()
)

display(
    missing_counts
    .rename("missing_rows")
    .reset_index()
    .rename(columns={"index": "column"})
)

if missing_counts.any():
    raise RuntimeError(
        "LightGBM merge still contains missing rows: "
        + str(missing_counts.to_dict())
    )

if len(all_predictions) != len(chronos_predictions):
    raise RuntimeError(
        "Merged row count changed unexpectedly: "
        f"{len(all_predictions)} != {len(chronos_predictions)}"
    )

all_predictions = all_predictions.drop(
    columns=["ml_label_check"]
)

print("Combined rows:", f"{len(all_predictions):,}")
print("LightGBM time alignment and merge validation passed.")
display(all_predictions.head())


In [ ]:
# Cell 18 - Unified model metrics / 统一模型指标

MODEL_COLUMNS = {
    "Seasonal naive - last week": "seasonal_naive_168h",
    "Chronos-2 - history only": "chronos_no_covariates_median",
    "Chronos-2 - calendar": "chronos_calendar_median",
    "Chronos-2 - calendar + weather": "chronos_calendar_weather_median",
    "LightGBM - calendar": "lightgbm_calendar",
    "LightGBM - calendar + weather": "lightgbm_calendar_weather",
}

def regression_metrics(frame, prediction_column):
    """Calculate regression evaluation metrics. / 计算回归评价指标。"""
    valid = frame[
        ["target", prediction_column]
    ].dropna()

    if valid.empty:
        return {
            "n": 0,
            "MAE": np.nan,
            "RMSE": np.nan,
            "R2": np.nan,
            "WAPE": np.nan,
            "bias": np.nan,
        }

    y = valid["target"].to_numpy(float)
    p = valid[prediction_column].to_numpy(float)

    denominator = np.abs(y).sum()

    return {
        "n": len(valid),
        "MAE": mean_absolute_error(y, p),
        "RMSE": np.sqrt(mean_squared_error(y, p)),
        "R2": r2_score(y, p),
        "WAPE": (
            np.abs(y - p).sum() / denominator
            if denominator > 0
            else np.nan
        ),
        "bias": np.mean(p - y),
    }

eval_df = all_predictions[
    all_predictions["is_dst_missing_hour"] == 0
].copy()


model_missing_summary = pd.DataFrame([
    {
        "model": model_name,
        "prediction_column": prediction_column,
        "missing_rows": int(
            eval_df[prediction_column].isna().sum()
        ),
        "valid_rows": int(
            eval_df[prediction_column].notna().sum()
        ),
    }
    for model_name, prediction_column in MODEL_COLUMNS.items()
])

display(model_missing_summary)

if (
    model_missing_summary["valid_rows"] == 0
).any():
    invalid_models = model_missing_summary.loc[
        model_missing_summary["valid_rows"] == 0,
        "model",
    ].tolist()

    raise RuntimeError(
        "These models have zero valid predictions and cannot "
        f"be compared: {invalid_models}"
    )

metric_rows = []

for model_name, prediction_column in MODEL_COLUMNS.items():
    metric_rows.append({
        "slice": "overall",
        "slice_value": "all",
        "model": model_name,
        **regression_metrics(
            eval_df,
            prediction_column,
        ),
    })

    for horizon in [1, 2, 3, 6, 12, 24]:
        subset = eval_df[
            eval_df["horizon"] == horizon
        ]

        metric_rows.append({
            "slice": "horizon",
            "slice_value": horizon,
            "model": model_name,
            **regression_metrics(
                subset,
                prediction_column,
            ),
        })

    for weather_type in [
        "dry",
        "rain_no_snow",
        "snow",
        "high_wind_only",
    ]:
        subset = eval_df[
            eval_df["weather_type"] == weather_type
        ]

        metric_rows.append({
            "slice": "weather_type",
            "slice_value": weather_type,
            "model": model_name,
            **regression_metrics(
                subset,
                prediction_column,
            ),
        })

model_metrics = pd.DataFrame(metric_rows)

overall_metrics = (
    model_metrics[
        model_metrics["slice"] == "overall"
    ]
    .sort_values("MAE")
)

horizon_metrics = (
    model_metrics[
        model_metrics["slice"] == "horizon"
    ]
    .sort_values(["slice_value", "MAE"])
)

weather_metrics = (
    model_metrics[
        model_metrics["slice"] == "weather_type"
    ]
    .sort_values(["slice_value", "MAE"])
)

print("Overall metrics")
display(overall_metrics)

print("Horizon metrics")
display(horizon_metrics)

print("Weather metrics")
display(weather_metrics)

model_metrics.to_csv(
    OUTPUT_DIR / "unified_model_metrics.csv",
    index=False,
)


In [ ]:
# Cell 19 - Paired weather ablation / 天气特征配对消融

PAIRS = {
    "Chronos-2": (
        "chronos_calendar_median",
        "chronos_calendar_weather_median",
    ),
    "LightGBM": (
        "lightgbm_calendar",
        "lightgbm_calendar_weather",
    ),
}

def bootstrap_ci(values):
    """Estimate a bootstrap confidence interval. / 使用 bootstrap 估计置信区间。"""
    values = np.asarray(values, dtype=float)
    values = values[np.isfinite(values)]

    if len(values) == 0:
        return np.nan, np.nan, np.nan

    rng = np.random.default_rng(RANDOM_STATE)
    means = np.empty(BOOTSTRAP_REPEATS)

    for index in range(BOOTSTRAP_REPEATS):
        means[index] = rng.choice(
            values,
            size=len(values),
            replace=True,
        ).mean()

    return (
        float(values.mean()),
        float(np.quantile(means, 0.025)),
        float(np.quantile(means, 0.975)),
    )

ablation_rows = []

for family, (calendar_column, weather_column) in PAIRS.items():
    working = eval_df[
        [
            "timestamp",
            "weather_type",
            "target",
            calendar_column,
            weather_column,
        ]
    ].dropna().copy()

    working["calendar_error"] = (
        working["target"]
        - working[calendar_column]
    ).abs()

    working["weather_error"] = (
        working["target"]
        - working[weather_column]
    ).abs()

    per_timestamp = (
        working
        .groupby(
            ["timestamp", "weather_type"],
            as_index=False,
        )
        .agg(
            calendar_MAE=("calendar_error", "mean"),
            weather_MAE=("weather_error", "mean"),
        )
    )

    per_timestamp["delta"] = (
        per_timestamp["weather_MAE"]
        - per_timestamp["calendar_MAE"]
    )

    for slice_name, subset in [
        ("all", per_timestamp),
        (
            "dry",
            per_timestamp[
                per_timestamp["weather_type"] == "dry"
            ],
        ),
        (
            "rain_no_snow",
            per_timestamp[
                per_timestamp["weather_type"] == "rain_no_snow"
            ],
        ),
        (
            "snow",
            per_timestamp[
                per_timestamp["weather_type"] == "snow"
            ],
        ),
    ]:
        mean_delta, ci_low, ci_high = bootstrap_ci(
            subset["delta"]
        )

        ablation_rows.append({
            "model_family": family,
            "weather_slice": slice_name,
            "unique_weather_hours": len(subset),
            "calendar_MAE": subset["calendar_MAE"].mean(),
            "weather_MAE": subset["weather_MAE"].mean(),
            "weather_minus_calendar_MAE": mean_delta,
            "CI_low": ci_low,
            "CI_high": ci_high,
        })

weather_ablation = pd.DataFrame(ablation_rows)

display(weather_ablation)

print(
    "Negative weather_minus_calendar_MAE means weather improved. "
    "If the 95% confidence interval crosses zero, the evidence "
    "is not stable enough for a strong claim."
)

weather_ablation.to_csv(
    OUTPUT_DIR / "weather_ablation_unique_timestamps.csv",
    index=False,
)


In [ ]:
# Cell 20 - Peak detection comparison / 高峰识别对比
# Time alignment / 时间对齐：每个模型和 horizon 先过滤 NaN / inf，再计算高峰指标

peak_rows = []

for model_name, prediction_column in MODEL_COLUMNS.items():
    raw_score = (
        eval_df[prediction_column]
        - eval_df["expected_pickup"]
    ) / eval_df["std_pickup"]

    # 将正负无穷转成 NaN，后面统一过滤。
    prediction_score = raw_score.replace(
        [np.inf, -np.inf],
        np.nan,
    )

    for horizon in [1, 2, 3, 6, 12, 24]:
        horizon_mask = eval_df["horizon"] == horizon

        # average_precision_score 不接受 NaN。
        # 同时确保标签、预测、基准均存在，且标准差严格大于 0。
        valid_mask = (
            horizon_mask
            & eval_df["is_high_peak"].notna()
            & eval_df[prediction_column].notna()
            & eval_df["expected_pickup"].notna()
            & eval_df["std_pickup"].notna()
            & (eval_df["std_pickup"] > 0)
            & prediction_score.notna()
        )

        n_total = int(horizon_mask.sum())
        n_valid = int(valid_mask.sum())
        n_excluded = n_total - n_valid

        if n_valid == 0:
            peak_rows.append({
                "model": model_name,
                "horizon": horizon,
                "n_total": n_total,
                "n_valid": 0,
                "n_excluded_nan_or_inf": n_excluded,
                "actual_peak_n": 0,
                "predicted_peak_n": 0,
                "precision": np.nan,
                "recall": np.nan,
                "F1": np.nan,
                "average_precision": np.nan,
            })
            continue

        y_true = eval_df.loc[
            valid_mask,
            "is_high_peak",
        ].astype(int)

        scores = prediction_score.loc[
            valid_mask
        ].astype(float)

        predicted_values = eval_df.loc[
            valid_mask,
            prediction_column,
        ].astype(float)

        expected_values = eval_df.loc[
            valid_mask,
            "expected_pickup",
        ].astype(float)

        y_pred = (
            (scores >= HIGH_Z_THRESHOLD)
            & (predicted_values >= MIN_PEAK_COUNT)
            & (expected_values >= MIN_EXPECTED_FOR_PEAK)
        ).astype(int)

        # AP 至少需要一个真实正例才具有可解释性。
        if y_true.sum() > 0:
            average_precision = average_precision_score(
                y_true,
                scores,
            )
        else:
            average_precision = np.nan

        peak_rows.append({
            "model": model_name,
            "horizon": horizon,
            "n_total": n_total,
            "n_valid": n_valid,
            "n_excluded_nan_or_inf": n_excluded,
            "actual_peak_n": int(y_true.sum()),
            "predicted_peak_n": int(y_pred.sum()),
            "precision": precision_score(
                y_true,
                y_pred,
                zero_division=0,
            ),
            "recall": recall_score(
                y_true,
                y_pred,
                zero_division=0,
            ),
            "F1": f1_score(
                y_true,
                y_pred,
                zero_division=0,
            ),
            "average_precision": average_precision,
        })

peak_metrics = pd.DataFrame(peak_rows)

display(peak_metrics)

print("Excluded rows by model:")
display(
    peak_metrics.groupby("model", as_index=False).agg(
        total_rows=("n_total", "sum"),
        valid_rows=("n_valid", "sum"),
        excluded_nan_or_inf=("n_excluded_nan_or_inf", "sum"),
    )
)

peak_metrics.to_csv(
    OUTPUT_DIR / "peak_detection_metrics.csv",
    index=False,
)


In [ ]:
# Cell 21 - LightGBM feature importance / LightGBM 特征重要性

feature_importance = pd.DataFrame({
    "feature": ML_WEATHER_FEATURES,
    "gain": (
        lightgbm_weather.booster_
        .feature_importance(importance_type="gain")
    ),
}).sort_values("gain", ascending=False)

display(feature_importance.head(30))

plot_df = feature_importance.head(25).sort_values("gain")

plt.figure(figsize=(10, 7))
plt.barh(
    plot_df["feature"],
    plot_df["gain"],
)
plt.title("LightGBM calendar + weather feature importance")
plt.tight_layout()
plt.show()

feature_importance.to_csv(
    OUTPUT_DIR / "lightgbm_feature_importance.csv",
    index=False,
)


In [ ]:
# Cell 22 - Model comparison charts / 模型对比图

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

overall_plot = overall_metrics.sort_values("MAE")

axes[0].barh(
    overall_plot["model"],
    overall_plot["MAE"],
)
axes[0].set_title("Overall MAE: lower is better")
axes[0].set_xlabel("Trips per H3-hour")

for model_name, subset in horizon_metrics.groupby("model"):
    subset = subset.copy()
    subset["h"] = pd.to_numeric(
        subset["slice_value"],
        errors="coerce",
    )
    subset = subset.dropna(
        subset=["h"]
    ).sort_values("h")

    axes[1].plot(
        subset["h"],
        subset["MAE"],
        marker="o",
        label=model_name,
    )

axes[1].set_title("MAE by forecast horizon")
axes[1].set_xlabel("Hours ahead")
axes[1].set_ylabel("MAE")
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
# Cell 23 - A 24-hour forecast example / 24 小时预测案例

example_origin = eval_df["forecast_origin"].min()
example_id = eval_df["id"].iloc[0]

example = eval_df[
    (eval_df["forecast_origin"] == example_origin)
    & (eval_df["id"] == example_id)
].sort_values("timestamp")

plt.figure(figsize=(14, 6))

plt.plot(
    example["timestamp"],
    example["target"],
    marker="o",
    label="Actual",
)

for model_name, prediction_column in MODEL_COLUMNS.items():
    plt.plot(
        example["timestamp"],
        example[prediction_column],
        label=model_name,
    )

plt.title(
    f"24-hour comparison - H3 {example_id} - "
    f"origin {example_origin}"
)
plt.ylabel("Completed non-shared pickups")
plt.xticks(rotation=35)
plt.legend(fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:
# Cell 24 - Export predictions and summary / 导出预测与总结

all_predictions.to_csv(
    OUTPUT_DIR / "all_model_predictions_2024.csv.gz",
    index=False,
    compression="gzip",
)

best = overall_metrics.iloc[0]

summary = {
    "python_executable": sys.executable,
    "python_version": sys.version,
    "full_h3_rows": int(rows_h3.loc[0, "rows_h3"]),
    "top_n_cells": TOP_N_CELLS,
    "weather_source": weather_source,
    "chronos_model": CHRONOS_MODEL_ID,
    "chronos_device": CHRONOS_DEVICE,
    "chronos_isolated_subprocess": True,
    "backtest_stride_hours": BACKTEST_STRIDE_HOURS,
    "backtest_origin_count": len(backtest_origins),
    "prediction_rows": len(all_predictions),
    "best_model_by_MAE": best["model"],
    "best_MAE": float(best["MAE"]),
}

with open(
    OUTPUT_DIR / "run_summary.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        summary,
        file,
        ensure_ascii=False,
        indent=2,
    )

print(json.dumps(summary, ensure_ascii=False, indent=2))


In [ ]:
# Cell 25 - Close MatrixOne connection / 关闭 MatrixOne 连接

try:
    conn.close()
    print("MatrixOne connection closed.")
except Exception as exc:
    print("Connection close warning:", repr(exc))


# Outputs and validation checklist / 输出与验证清单

The notebook exports the following reproducible artifacts:  
本 Notebook 导出以下可复现实验产物：

1. `overall_metrics` and `horizon_metrics` for model accuracy. / `overall_metrics` 与 `horizon_metrics`：模型准确率。
2. `weather_ablation` for paired weather-feature comparison. / `weather_ablation`：天气特征配对对比。
3. `peak_metrics` for high-demand event detection. / `peak_metrics`：高需求事件识别。
4. LightGBM feature-importance and model-MAE charts. / LightGBM 特征重要性图与模型 MAE 对比图。
5. A 24-hour forecast example and `run_summary.json`. / 一个 24 小时预测案例与 `run_summary.json`。

The isolated worker reports `CHRONOS_WORKER_SUCCESS` and return code `0` after a successful run. Chunk files allow an interrupted full backtest to resume without repeating completed forecast origins.  
隔离进程成功完成后会输出 `CHRONOS_WORKER_SUCCESS` 和返回码 `0`。分块文件使中断的完整回测能够继续运行，而不重复已完成的预测起点。
